# 🌪️ NCSSM WindTunnel lab · 2D Lattice Boltzmann Wind Tunnel
**North Carolina School of Science and Mathematics**  
*Real-Time Computational Aerodynamics · LBM D2Q9 · GPU Acceleration & WebGL Visualization*

---

### ⚡ Instructions for Google Colab:
1. **Enable Free GPU Acceleration (Recommended)**:
   * Go to **Runtime** ➔ **Change runtime type** in the top menu.
   * Select **T4 GPU** (or any available GPU) under *Hardware accelerator* and click **Save**.
2. **Run Step 1 below**: Clones the repository, installs dependencies (`fastapi`, `uvicorn`, `cupy-cuda12x`), and sets up the Cloudflare Tunnel.
3. **Run Step 2 below**: Starts the simulation engine and generates a secure public link.
4. Click **`🚀 Open WindTunnel lab ↗`** to open the interactive wind tunnel in your browser!

In [ ]:
# @title Step 1: Clone Repository & Setup Environment { display-mode: "form" }
import os, sys

print("🚀 Setting up NCSSM WindTunnel lab environment...")

# Clone repository if not already present in Colab workspace
if not os.path.exists("2D_WindTunnel") and not os.path.exists("backend"):
    !git clone https://github.com/fsantibanezc/WindTunnel.git 2D_WindTunnel
    %cd 2D_WindTunnel
elif os.path.exists("2D_WindTunnel"):
    %cd 2D_WindTunnel

# Install backend dependencies and CuPy for GPU execution
!pip install -q fastapi "uvicorn[standard]" numpy "cupy-cuda12x>=13.0"

# Install Cloudflare Tunnel client for bidirectional WebSocket streaming
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i -E cloudflared-linux-amd64.deb > /dev/null 2>&1 || true

print("✅ Setup complete and ready to launch!")

In [ ]:
# @title Step 2: Launch WindTunnel lab Server & Tunnel { display-mode: "form" }
import subprocess
import time
import re
from IPython.display import display, HTML

# Terminate any prior server or tunnel processes
!pkill -f "backend/server.py" > /dev/null 2>&1 || true
!pkill -f "cloudflared" > /dev/null 2>&1 || true
time.sleep(1)

print("🌪️ Starting WindTunnel lab simulation server...")
server_process = subprocess.Popen(
    ["python3", "backend/server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
time.sleep(2.5)

print("🔗 Connecting Cloudflare Tunnel for live WebSocket streaming...")
tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
for _ in range(40):
    line = tunnel_process.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.5)

if tunnel_url:
    html_banner = f"""
    <div style="background: linear-gradient(135deg, #09121d, #142236); border: 1.5px solid #00e5ff; border-radius: 12px; padding: 22px; margin: 12px 0; color: #ffffff; box-shadow: 0 8px 32px rgba(0, 229, 255, 0.25); font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
      <div style="display: flex; align-items: center; gap: 16px; margin-bottom: 12px;">
        <div style="background: #ffffff; border-radius: 8px; padding: 4px; display: flex; align-items: center; justify-content: center; width: 48px; height: 48px; box-shadow: 0 2px 8px rgba(0,0,0,0.3);">
          <img src="{tunnel_url}/static/ncssm_logo.jpg" alt="NCSSM Logo" style="width: 40px; height: 40px; object-fit: contain;" />
        </div>
        <div>
          <h2 style="margin: 0; color: #00e5ff; font-size: 20px; letter-spacing: 0.5px;">NCSSM WindTunnel lab</h2>
          <p style="margin: 2px 0 0; color: #94a3b8; font-size: 13px;">2D Lattice Boltzmann Wind Tunnel · Live WebGL & GPU</p>
        </div>
      </div>
      <p style="color: #cbd5e1; font-size: 14px; margin: 14px 0 16px;">
        Your high-performance simulation instance is live and streaming at full GPU speed. Click the button below to launch the wind tunnel in a new tab:
      </p>
      <a href="{tunnel_url}" target="_blank" style="display: inline-block; background: linear-gradient(135deg, #00e5ff, #c084fc); color: #061018; font-weight: 700; font-size: 15px; padding: 12px 26px; border-radius: 8px; text-decoration: none; box-shadow: 0 4px 18px rgba(0, 229, 255, 0.4);">
        🚀 Open WindTunnel lab ↗
      </a>
      <div style="margin-top: 14px; font-size: 12px; color: #64748b;">
        Direct link: <a href="{tunnel_url}" target="_blank" style="color: #38bdf8;">{tunnel_url}</a>
      </div>
    </div>
    """
    display(HTML(html_banner))
else:
    print("❌ Could not obtain public tunnel URL. Check execution logs above.")
